In [1]:
pip install pandas scikit-learn sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 63.8 MB/s eta 0:00:00


In [6]:
import pandas as pd

# Example CSV with columns: 'title' and 'description'
df = pd.read_csv("data (1).csv")

# Combine title + description for richer embeddings
df['text'] = df['title'].fillna('') + " " + df['description'].fillna('')


In [7]:
from sentence_transformers import SentenceTransformer

# Use a small but effective model; you can also try 'all-MiniLM-L6-v2'
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all text
embeddings = model.encode(df['text'].tolist(), show_progress_bar=True)


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [21]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

class TextRecommender:
    def __init__(self, df, text_column='CombinedInfo', model_name='all-MiniLM-L6-v2', index_path="faiss_index.index"):
        """
        df: pandas DataFrame containing your data
        text_column: column to encode (title + description)
        index_path: path to save the FAISS index
        """
        self.df = df.copy()
        self.text_column = text_column
        self.index_path = index_path

        # Initialize model
        self.model = SentenceTransformer(model_name)

        # Generate embeddings
        print("Generating embeddings...")
        self.embeddings = self.model.encode(self.df[self.text_column].tolist(), convert_to_tensor=False, show_progress_bar=True)
        self.embeddings = np.array(self.embeddings, dtype=np.float32)

        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(self.embeddings)

        # Build FAISS index
        self.d = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(self.d)  # inner product = cosine similarity
        self.index.add(self.embeddings)
        print(f"FAISS index created with {self.index.ntotal} items.")

        # Save FAISS index
        faiss.write_index(self.index, self.index_path)
        print(f"FAISS index saved to {self.index_path}")

    def recommend_by_description(self, user_query, top_k=5, min_similarity=0.0):
        """
        user_query: string to search for
        top_k: number of results to return
        min_similarity: filter out results below this similarity score (0 to 1)
        """
        # Encode and normalize query
        query_emb = self.model.encode([user_query], convert_to_tensor=False)
        query_emb = np.array(query_emb, dtype=np.float32)
        faiss.normalize_L2(query_emb)

        # Search FAISS
        distances, indices = self.index.search(query_emb, top_k)

        # Build recommendations DataFrame
        recommendations = self.df.iloc[indices[0]].copy().reset_index(drop=True)
        recommendations['DescriptionSimilarity'] = distances[0]

        # Filter by min_similarity
        recommendations = recommendations[recommendations['DescriptionSimilarity'] >= min_similarity]

        return recommendations[['event_id','DescriptionSimilarity']]




# Initialize recommender (this will automatically save the FAISS index)
recommender = TextRecommender(df, index_path="faiss_index.index")

# Get top 5 recommendations
query = "Central Park Bird Adventure"
results = recommender.recommend_by_description(query, top_k=5)
print(results)


Generating embeddings...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

FAISS index created with 314 items.
FAISS index saved to faiss_index.index
        event_id  DescriptionSimilarity
0  1379380045849               0.703665
1  1571073185189               0.532107
2  1357265751399               0.506922
3  1568645463809               0.487938
4  1560210965999               0.469917
